# Regression.ml (in progress)

In this module, we assume that the conditional outcome model in each source domain $1 \le l \le L$ admits any form:

$$
Y^{(l)} = f^{(l)}(X^{(l)}) + \varepsilon^{(l)}, 
\qquad \text{with} \quad \mathbb{E}[\varepsilon^{(l)}|X^{(l)}] = 0,
$$

which implies
$$
\mathbb{E}[Y^{(l)}|X^{(l)}] = f^{(l)}(X^{(l)}) X^{(l)}.
$$

To learn a **robust prediction model** under domain shifts, the CGDRO framework solves the following minimax optimization problem:

$$
f^* = \arg\min_{f \in \mathcal{F}}
\max_{\mathbf{T} \in \mathcal{C}}
\mathbb{E}_{(X, Y) \sim \mathbf{T}} \ell(X, Y; f),

$$

where $\mathcal{C}$ is the **uncertainty class** over possible target distributions, as defined in the [Introduction](../setup/intro.md).

We can import `Regression.ml` module by the code below:

In [ ]:
from Regression import ml

Now we give an example showing how to implement `Regression.ml` with three different loss functions:

- Reward-based loss  
- Squared loss  
- Regret-based loss  

---

## Module Arguments & Outputs 

#### Regression.ml
- `f_learner` (str, optional): method used to fit outcome models on each source. Defaults to 'xgb'. Including `linear`, `xgb`, `mlp`, and `rf`.
- `w_learner` (str, optional): method used to fit density models on each source. Defaults to 'xgb'. Including `linear`, `xgb`, and `kliep`.
- `seed` (int, optional): random seed for data-splitting. Defaults to 123.
- `verbose` (bool, optional): whether to print out the fitting information. Defaults to False.


Built-in functions in `Regression.ml`:

| BUilt-in Functions     | Description                                                                 |
|------------|-----------------------------------------------------------------------------|
| `fit()`    | Fit robust machine learning regression in the target domain.                |
| `predict()`| Make robust prediction in the target domain.                                |




#### fit()
**Arguments:**
- `X_list` (list): list of feature matrices on each source domain.
- `y_list` (list): list of label arrays on each source domain.
- `X0` (array, optional): feature matrix on the target domain. If None, use the pooled source data as the target data. Defaults to None.
- `loss_type` (str, optional): type of the loss function used to compute the optimal aggregation weights. Options include 'reward' (default), 'squaredloss', and 'regret'. Defaults to 'reward'.
- `bias_correct` (bool, optional): whether to use the bias-corrected estimator of the Gamma matrix. Defaults to True.
- `priors` (tuple, optional): prior information on the aggregation weights, given as (prior_weight, rho), where prior_weight is the prior weight vector and rho is the radius of the L2-norm ball around prior_weight. If None, no prior information is used. Defaults to None.

            
**Outputs:**
enabled the following attributes:

- `weight_`: CGDRO aggregated weights of the source domains.



#### predict()
**Outputs:**
- `pred` : linear prediction in the target domain.

---

## Example

### Data Generating Process

In this example, we generate a non-linear multi-source domain data with $3$ domains, putting $10,000$ samples on each source domain and $100,000$ samples on the target domain. The dimension of the parameters is $p=5$,

In [ ]:
# number of source groups = 3, each with 10000 samples, and 100000 target samples
# dimension p = 5
# sigma: source group 1,3: 0.5; source group 2: 3.
data = DataContainerSimu_Nonlinear_reg(n=10000, N=100000)
data.generate_funcs_list(L=3, seed=0)
data.generate_data()

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target

### Implementation & Prediction

We implement three loss functions by `Regression.ml`, including `reward`, `squaredloss`, and `regret`. Geometrically, `reward`: $f^∗$ is the point closest to the original within the convex hull of ${f(l)}_{l\in[L]}$; `squaredloss`: $f^{sq}$ corresponds to the source model with the largest noise level with the highest noise level when this noise is substantially higher than that in other sources; `regret`: $f^{reg}$ is the center of the smallest circle enclosing all individual source models.


![Loss Types](../assets/loss_type.png)

#### loss_type =  reward

We define the loss as  
$\ell(X,Y;f) = (Y -  f(X))^2 - Y^2$.

Under this choice, the minimax problem becomes

$$
\begin{aligned}
f^*
&= \arg\min_{f \in \mathcal{F}} \max_{\mathbf{T}\in \mathcal{C}}
\mathbb{E}_{\mathbf{T}}\!\left[(Y-f(X))^2 - Y^2\right] \\[2mm]
&= \arg\max_{f \in \mathcal{F}} \min_{\mathbf{T}\in \mathcal{C}}
\mathbb{E}_{\mathbf{T}}\!\left[Y^2 - (Y-f(X))^2\right].
\end{aligned}
$$

The right-hand side shows that  $\mathbb{E}_{\mathbf{T}}\!\left[Y^2 - (Y - f(X))^2\right]$  can be interpreted as the **explained variance** of $Y$ by the predictor $f(X)$ (assuming $Y$ is centered).  
Hence, the CGDRO model maximizes the **worst-case explained variance** across all possible target distributions in $\mathcal{C}$.


**Proposition**

The CGDRO model $f^*$ with reward-based loss admits the closed form:

$$
f^* = \sum_{l=1}^L q_l^* \, f^{(l)},
\qquad
q^* = \arg\min_{q \in \Delta^L} q^\top \Gamma q,
$$

where $\Gamma \in \mathbb{R}^{L \times L}$ is defined by  

$$
\Gamma_{k,l} =  \mathbb{E}_{\mathbf{Q}}[f^{(k)}(X) f^{(l)}(X)], 
\quad k,l \in [L].
$$

This result implies that $f^*$ is a **convex combination** of the source-specific conditional models $\{f^{(l)}\}$,  
with weights $q^*$ minimizing the second-moment matrix of the target covariates.

In [ ]:
## First announcing the module
## Then calling the functions fit() 
drol = ml(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='reward')


In [ ]:
drol.weight_

array([0.23619293, 0.29104051, 0.47276655])

In [ ]:
drol.predict()

array([-1.37911566, -1.01748387,  1.44855369, ..., -1.79589733,
       -0.42999676,  2.98552634])

In [ ]:
# Geometry view: the convex hull of the source coefficients (cloesest point to the origin)
pred_source = drol.pred_full_mat.T
pred_ch, w_ch = nearest_on_convex_hull(pred_source)
print("Predictions on convex hull:", pred_ch)
print("Weights:", w_ch)


Predictions on convex hull: [-1.38436167 -1.00976801  1.47364845 ... -1.81097766 -0.44569888
  2.96259133]
Weights: [0.23864968 0.28245441 0.4788959 ]


#### loss_type =  squaredloss

We may also choose the standard squared loss
$\ell(X,Y;f) = (Y - f(X))^2$.

Then the minimax problem becomes

$$
f_{\text{sq}} =
\arg\min_{f} \max_{\mathbf{T} \in \mathcal{C}}
\mathbb{E}_{(X,Y)\sim \mathbf{T}} (Y - f(X))^2.
$$

Define the noise level in each source domain:

$$
(\sigma^{(l)})^2 = \mathbb{E}\!\left[(\varepsilon_i^{(l)})^2 \mid X_i^{(l)}\right],
\qquad
\varepsilon_i^{(l)} = Y_i^{(l)} - f^{(l)}(X_i^{(l)}).
$$

Let $\boldsymbol{\sigma}^2 = ((\sigma^{(1)})^2, \ldots, (\sigma^{(L)})^2)$.

**Proposition**

The CGDRO model $f_{\text{sq}}$ under squared loss admits the closed form:

$$
f_{\text{sq}} = \sum_{l=1}^L q_l^{\text{sq}} \, f^{(l)},
\qquad
q_{\text{sq}} = \arg\min_{q \in \Delta^L}
q^\top \Gamma q - q^\top (\gamma + \boldsymbol{\sigma}^2),
$$

where $\Gamma \in \mathbb{R}^{L\times L}$ is defined as

$$
\Gamma_{k,l} =  \mathbb{E}_{\mathbf{Q}}[f^{(k)}(X) f^{(l)}(X)], 
\quad k,l \in [L].
$$

and $\gamma \in \mathbb{R}^L$ is the diagonal of $\Gamma$:
$\gamma_l = \Gamma_{l,l}$ for $l \in [L]$.


In [ ]:
drol = ml(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='squaredloss')

## time cost: 8.7s

In [ ]:
drol.weight_

array([0., 1., 0.])

In [ ]:
drol.predict()

array([-0.8600474 , -1.5655055 , -0.60189897, ..., -0.58915877,
        0.9634735 ,  5.21788502])

In [ ]:
# Geometry view: the sufficiently large noise group dominates
pred_source = drol.pred_full_mat.T
pred_source[1]

array([-0.8600474 , -1.5655055 , -0.60189897, ..., -0.58915877,
        0.9634735 ,  5.21788502])

#### loss_type =  regret

The **regret** is defined as

$$
\mathrm{Regret}_{\mathbf{T}}(f)
:= \mathbb{E}_{\mathbf{T}}\!\left[(Y-f(X))^2\right]
- \inf_{f'} \mathbb{E}_{\mathbf{T}}\!\left[(Y - f'(X))^2\right].
$$

It measures the **excess risk** of $f$ compared to the optimal model for distribution $\mathbf{T}$.  
The CGDRO formulation then seeks to minimize the **worst-case regret**:

$$
f_{\text{reg}} =
\arg\min_{f \in \mathcal{F}}
\max_{\mathbf{T} \in \mathcal{C}}
\mathrm{Regret}_{\mathbf{T}}(f).
$$

**Proposition**

The CGDRO model $f_{\text{reg}}$ under regret function admits the closed form:

$$
f_{\text{reg}} = \sum_{l=1}^L q_l^{\text{reg}} \, f^{(l)},
\qquad
q_{\text{reg}} = \arg\min_{q \in \Delta^L}
q^\top \Gamma q - q^\top \gamma,
$$

where $\Gamma \in \mathbb{R}^{L\times L}$ is defined as


$$
\Gamma_{k,l} =  \mathbb{E}_{\mathbf{Q}}[f^{(k)}(X) f^{(l)}(X)], 
\quad k,l \in [L].
$$

and $\gamma \in \mathbb{R}^L$ is the diagonal of $\Gamma$:
$\gamma_l = \Gamma_{l,l}$ for $l \in [L]$.


In [ ]:
drol = ml(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='regret')

## time cost: 8.7s

In [ ]:
drol.weight_

array([0.41122666, 0.21230474, 0.3764686 ])

In [ ]:
drol.predict()

array([-1.82018649, -1.35423715,  1.57969483, ..., -1.75827626,
       -1.01742939,  1.22626068])

In [ ]:
# Geometry view: the center of the minimum enclosing ball of the source coefficients
pred_source = drol.pred_full_mat.T
pred_cr, r_cr, w_cr = circumcenter_3vectors(pred_source)
print("Predictions on center of minimum enclosing ball:", pred_cr)
print("Weights:", w_cr)

Predictions on center of minimum enclosing ball: [ 1.19534202 -0.7734592   0.13471152 ...  1.21178966  0.54131003
 -0.07881488]
Weights: [0.39353541 0.26546146 0.34100313]
